In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import pipeline

In [2]:
df = pd.read_csv("../data/clean_hr_data.csv")

In [3]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

In [4]:
# BOW method 
# You fit and transform on the column
# you only transform the y1 and y2
vectorizer_bow = CountVectorizer()
X = vectorizer_bow.fit_transform(df['clean_job_title'])
y1 = vectorizer_bow.transform([measure_rank_1])
y2 = vectorizer_bow.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)

df["similarity_1_bow"] = score_1.flatten()
df[df["similarity_1_bow"]>0.3][["clean_job_title", "similarity_1_bow"]].sort_values(
    by="similarity_1_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_1_bow
21,aspiring human resources manager seek internsh...,0.745356
47,aspiring human resources manager | graduating ...,0.629941
6,student humber college aspiring human resource...,0.612372
5,aspiring human resources specialist greater ne...,0.577350
2,aspiring human resources professional raleigh ...,0.577350
12,aspiring human resources management student se...,0.547723
20,business management major aspiring human resou...,0.547723
30,aspiring human resources professional | energe...,0.522233
22,human resources professional greater boston area,0.471405
36,human resources management major milpitas cali...,0.471405


In [5]:
df["similarity_2_bow"] = score_2.flatten()
df[df["similarity_2_bow"]>0.3][["clean_job_title", "similarity_2_bow"]].sort_values(
    by="similarity_2_bow",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bow
21,aspiring human resources manager seek internsh...,0.730297
47,aspiring human resources manager | graduating ...,0.617213
13,seek human resources opportunities chicago ill...,0.577350
22,human resources professional greater boston area,0.577350
36,human resources management major milpitas cali...,0.577350
37,director human resources ey greater atlanta ...,0.534522
26,human resources generalist schwan amerika birl...,0.534522
15,human resources staffing recruiting profession...,0.500000
46,seek human resources position las vegas nevada...,0.500000
6,student humber college aspiring human resource...,0.500000


In [6]:
# Tfidf method
vectorizer_tfidf = TfidfVectorizer()
X = vectorizer_tfidf.fit_transform(df['clean_job_title'])
y1 = vectorizer_tfidf.transform([measure_rank_1])
y2 = vectorizer_tfidf.transform([measure_rank_2])

score_1 = cosine_similarity(y1, X)
score_2 = cosine_similarity(y2, X)


df["similarity_1_tfidf"] = score_1.flatten()
df[df["similarity_1_tfidf"]>0.3][["clean_job_title", "similarity_1_tfidf"]].sort_values(
    by="similarity_1_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_1_tfidf
21,aspiring human resources manager seek internsh...,0.547299
5,aspiring human resources specialist greater ne...,0.416505
2,aspiring human resources professional raleigh ...,0.410821
12,aspiring human resources management student se...,0.401339
6,student humber college aspiring human resource...,0.383511
20,business management major aspiring human resou...,0.361899
47,aspiring human resources manager | graduating ...,0.321200
30,aspiring human resources professional | energe...,0.319760


In [7]:
df["similarity_2_tfidf"] = score_2.flatten()
df[df["similarity_2_tfidf"]>0.3][["clean_job_title", "similarity_2_tfidf"]].sort_values(
    by="similarity_2_tfidf",
    ascending=False
).head(10)

,clean_job_title,similarity_2_tfidf
21,aspiring human resources manager seek internsh...,0.491453
22,human resources professional greater boston area,0.339593


In [8]:
#%pip install gensim

model = api.load('glove-wiki-gigaword-100')


# apply has the tendency to work on the rows automatically 
def get_title_vector(row):
    words = row.split()

    vectors = []

    for word in words:
        if word in model:
            vectors.append(model[word])

    return np.mean(vectors, axis=0)


    
df['vector_for_modeling'] = df["clean_job_title"].apply(get_title_vector)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)


X_glove = np.vstack(df["vector_for_modeling"])
# to make it work you need to use vstack function
score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_glove
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_glove
)

df["similarity_1_glove"] = score_1.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_1_glove"]>0.3][["clean_job_title", "similarity_1_glove"]].sort_values(
    by="similarity_1_glove",
    ascending=False
).head(10)

,clean_job_title,similarity_1_glove
21,aspiring human resources manager seek internsh...,0.873273
22,human resources professional greater boston area,0.832801
13,seek human resources opportunities chicago ill...,0.829122
24,aspiring human resources professional | passio...,0.829009
37,director human resources ey greater atlanta ...,0.817207
5,aspiring human resources specialist greater ne...,0.816461
8,seek human resources hris generalist positions...,0.815399
12,aspiring human resources management student se...,0.814947
32,human resources professional world leader gis ...,0.814346
36,human resources management major milpitas cali...,0.794404


In [9]:
df["similarity_2_glove"] = score_2.flatten()
# turns a multi-dimensional NumPy array into a one-dimensional array
df[df["similarity_2_glove"]>0.3][["clean_job_title", "similarity_2_glove"]].sort_values(
    by="similarity_2_glove",
    ascending=False
).head(10)


,clean_job_title,similarity_2_glove
21,aspiring human resources manager seek internsh...,0.887125
13,seek human resources opportunities chicago ill...,0.886082
8,seek human resources hris generalist positions...,0.875451
42,seek human resources opportunity open travel...,0.872547
22,human resources professional greater boston area,0.848357
37,director human resources ey greater atlanta ...,0.840632
36,human resources management major milpitas cali...,0.840518
12,aspiring human resources management student se...,0.814407
24,aspiring human resources professional | passio...,0.809889
16,human resources specialist luxottica greater n...,0.809206


In [10]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def get_title_vector_bert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )
# pt is py_torch, truncation for fitting what the model allow in size, padding makes the size the same
    with torch.no_grad():
        outputs = bert(**inputs)


    vector = outputs.last_hidden_state.mean(dim=1)

    return vector.squeeze().numpy()
# make the tensor a numpy array with one dimension 
df['bert_for_modeling'] = df["clean_job_title"].apply(get_title_vector_bert)
y1_vector = get_title_vector(measure_rank_1)
y2_vector = get_title_vector(measure_rank_2)



X_bert = np.vstack(df["bert_for_modeling"])

y1_vector = get_title_vector_bert(measure_rank_1)
y2_vector = get_title_vector_bert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bert
)

df["similarity_1_bert"] = score_1.flatten()

df[df["similarity_1_bert"]>0.3][["clean_job_title", "similarity_1_bert"]].sort_values(
    by="similarity_1_bert",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,clean_job_title,similarity_1_bert
13,seek human resources opportunities chicago ill...,0.756244
21,aspiring human resources manager seek internsh...,0.751372
30,aspiring human resources professional | energe...,0.738730
6,student humber college aspiring human resource...,0.734590
14,experienced retail manager aspire human resour...,0.731170
20,business management major aspiring human resou...,0.716073
12,aspiring human resources management student se...,0.715341
15,human resources staffing recruiting profession...,0.711508
51,director administration excellence logging kat...,0.710626
32,human resources professional world leader gis ...,0.706642


In [11]:
df["similarity_2_bert"] = score_2.flatten()

df[df["similarity_2_bert"]>0.3][["clean_job_title", "similarity_2_bert"]].sort_values(
    by="similarity_2_bert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bert
13,seek human resources opportunities chicago ill...,0.822273
15,human resources staffing recruiting profession...,0.704242
21,aspiring human resources manager seek internsh...,0.698324
51,director administration excellence logging kat...,0.695820
8,seek human resources hris generalist positions...,0.693475
40,seek employment opportunity customer service p...,0.693326
3,people development coordinator ryan denton texas,0.686898
37,director human resources ey greater atlanta ...,0.682326
22,human resources professional greater boston area,0.681378
14,experienced retail manager aspire human resour...,0.670993


In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

sbert = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)


def get_title_vector_sbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = sbert(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Mean pooling, ignoring padding tokens
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["sbert_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_sbert
)


s_bert = np.vstack(df["sbert_for_modeling"])

y1_vector = get_title_vector_sbert(measure_rank_1)
y2_vector = get_title_vector_sbert(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    s_bert
)

score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    s_bert
)

df["similarity_1_sbert"] = score_1.flatten()

df[df["similarity_1_sbert"]>0.3][["clean_job_title", "similarity_1_sbert"]].sort_values(
    by="similarity_1_sbert",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,clean_job_title,similarity_1_sbert
2,aspiring human resources professional raleigh ...,0.842901
5,aspiring human resources specialist greater ne...,0.787457
47,aspiring human resources manager | graduating ...,0.776213
30,aspiring human resources professional | energe...,0.726663
21,aspiring human resources manager seek internsh...,0.717666
12,aspiring human resources management student se...,0.710845
46,seek human resources position las vegas nevada...,0.708507
24,aspiring human resources professional | passio...,0.689745
13,seek human resources opportunities chicago ill...,0.678825
6,student humber college aspiring human resource...,0.675802


In [13]:
df["similarity_2_sbert"] = score_2.flatten()

df[df["similarity_2_sbert"]>0.3][["clean_job_title", "similarity_2_sbert"]].sort_values(
    by="similarity_2_sbert",
    ascending=False
).head(10)

,clean_job_title,similarity_2_sbert
46,seek human resources position las vegas nevada...,0.765440
2,aspiring human resources professional raleigh ...,0.749284
5,aspiring human resources specialist greater ne...,0.713229
13,seek human resources opportunities chicago ill...,0.712608
48,human resources generalist loparex raleigh dur...,0.712407
8,seek human resources hris generalist positions...,0.712170
22,human resources professional greater boston area,0.660824
21,aspiring human resources manager seek internsh...,0.656847
47,aspiring human resources manager | graduating ...,0.656618
30,aspiring human resources professional | energe...,0.650934


In [14]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)




X_e5 = np.vstack(df["e5_for_modeling"])

y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)


score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

df["similarity_1_5e"] = score_1.flatten()

df[df["similarity_1_5e"]>0.3][["clean_job_title", "similarity_1_5e"]].sort_values(
    by="similarity_1_5e",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,clean_job_title,similarity_1_5e
5,aspiring human resources specialist greater ne...,0.901982
21,aspiring human resources manager seek internsh...,0.897031
2,aspiring human resources professional raleigh ...,0.894147
47,aspiring human resources manager | graduating ...,0.894092
24,aspiring human resources professional | passio...,0.884487
12,aspiring human resources management student se...,0.882906
30,aspiring human resources professional | energe...,0.879660
13,seek human resources opportunities chicago ill...,0.874558
15,human resources staffing recruiting profession...,0.864990
20,business management major aspiring human resou...,0.862209


In [15]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_e5
)

df["similarity_2_5e"] = score_2.flatten()

df[df["similarity_2_5e"]>0.3][["clean_job_title", "similarity_2_5e"]].sort_values(
    by="similarity_2_5e",
    ascending=False
).head(10)

,clean_job_title,similarity_2_5e
13,seek human resources opportunities chicago ill...,0.907182
46,seek human resources position las vegas nevada...,0.894789
21,aspiring human resources manager seek internsh...,0.885686
8,seek human resources hris generalist positions...,0.879234
42,seek human resources opportunity open travel...,0.875551
47,aspiring human resources manager | graduating ...,0.873214
12,aspiring human resources management student se...,0.870509
5,aspiring human resources specialist greater ne...,0.866386
24,aspiring human resources professional | passio...,0.862875
2,aspiring human resources professional raleigh ...,0.858127


In [16]:
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

bge = AutoModel.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


def get_title_vector_bge(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = bge(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy() 


df["bge_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_bge
)


X_bge = np.vstack(df["bge_for_modeling"])

y1_vector = get_title_vector_bge(measure_rank_1)
y2_vector = get_title_vector_bge(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_bge
)

df["similarity_1_bge"] = score_1.flatten()
# Chain the filtering and sorting together
df[df["similarity_1_bge"] > 0.3][["clean_job_title", "similarity_1_bge"]].sort_values(
    by="similarity_1_bge",
    ascending=False
).head(10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,clean_job_title,similarity_1_bge
2,aspiring human resources professional raleigh ...,0.815475
5,aspiring human resources specialist greater ne...,0.800074
21,aspiring human resources manager seek internsh...,0.787040
24,aspiring human resources professional | passio...,0.784607
30,aspiring human resources professional | energe...,0.764258
12,aspiring human resources management student se...,0.763715
47,aspiring human resources manager | graduating ...,0.763419
20,business management major aspiring human resou...,0.756249
13,seek human resources opportunities chicago ill...,0.754647
6,student humber college aspiring human resource...,0.721071


In [17]:
score_2 = cosine_similarity(
    y2_vector.reshape(1, -1),
    X_bge
)

df["similarity_2_bge"] = score_2.flatten()

df[df["similarity_2_bge"] > 0.3][["clean_job_title", "similarity_2_bge"]].sort_values(
    by="similarity_2_bge",
    ascending=False
).head(10)

,clean_job_title,similarity_2_bge
13,seek human resources opportunities chicago ill...,0.839602
46,seek human resources position las vegas nevada...,0.789876
8,seek human resources hris generalist positions...,0.774203
2,aspiring human resources professional raleigh ...,0.770838
21,aspiring human resources manager seek internsh...,0.766566
5,aspiring human resources specialist greater ne...,0.751435
15,human resources staffing recruiting profession...,0.751198
42,seek human resources opportunity open travel...,0.742584
22,human resources professional greater boston area,0.738667
24,aspiring human resources professional | passio...,0.725922


In [18]:
df

,id,job_title,location,connection,relevance,clean_job_title,similarity_1_bow,similarity_2_bow,similarity_1_tfidf,similarity_2_tfidf,...,similarity_2_bert,sbert_for_modeling,similarity_1_sbert,similarity_2_sbert,e5_for_modeling,similarity_1_5e,similarity_2_5e,bge_for_modeling,similarity_1_bge,similarity_2_bge
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,1,2019 c.t. bauer college business graduate magn...,0.308607,0.377964,0.099687,0.159870,...,0.551268,"[-0.34165978, 0.071470454, 0.012781473, -0.018...",0.390027,0.282561,"[-0.5084922, 0.34237465, 0.5676787, 0.05718790...",0.820726,0.793657,"[-0.40054908, 0.3582064, 0.023666572, -0.32824...",0.610130,0.579220
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500,0,native english teacher epik english program ko...,0.000000,0.000000,0.000000,0.000000,...,0.575211,"[-0.007482983, 0.04508299, 0.2542144, -0.04180...",0.148046,0.141916,"[-0.456203, 0.33561006, 0.21070337, -0.0901796...",0.763373,0.745414,"[-0.16465878, 0.20876308, 0.69340354, -0.24499...",0.477236,0.465852
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,1,aspiring human resources professional raleigh ...,0.577350,0.471405,0.410821,0.256167,...,0.632049,"[0.03896105, 0.07679981, -0.268583, 0.6749405,...",0.842901,0.749284,"[-0.65419954, 0.20865859, 0.11589524, 0.170229...",0.894147,0.858127,"[-0.27730662, 0.7022705, 0.0535323, -0.2380222...",0.815475,0.770838
3,4,People Development Coordinator at Ryan,"Denton, Texas",500,0,people development coordinator ryan denton texas,0.000000,0.000000,0.000000,0.000000,...,0.686898,"[-0.39900938, -0.07422431, -0.068586335, 0.074...",0.391778,0.384026,"[-0.80403244, 0.4648779, 0.4698457, -0.1501685...",0.806019,0.797369,"[-0.10203578, -0.112945706, 0.14424647, -0.425...",0.582874,0.615213
4,5,Advisory Board Member at Celal Bayar University,"Izmir, Turkiye",500,0,advisory board member celal bayar university i...,0.000000,0.000000,0.000000,0.000000,...,0.329493,"[-0.29537693, 0.201363, -0.009500056, -0.00230...",0.243675,0.222491,"[-0.44008344, 0.26205158, 0.22837786, -0.06000...",0.743727,0.745177,"[0.0032356717, 0.41211772, 0.40210912, -0.4016...",0.453267,0.442491
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,1,aspiring human resources specialist greater ne...,0.577350,0.471405,0.416505,0.259712,...,0.639155,"[0.09876519, -0.10893437, 0.022738159, 0.52238...",0.787457,0.713229,"[-0.61147654, 0.3599533, 0.3801304, 0.16546346...",0.901982,0.866386,"[0.2279015, 0.38759235, 0.5125312, -0.3389072,...",0.800074,0.751435
6,7,Student at Humber College and Aspiring Human R...,Kanada,61,1,student humber college aspiring human resource...,0.612372,0.500000,0.383511,0.239138,...,0.641753,"[0.08145559, -0.12828213, 0.133419, 0.34408268...",0.675802,0.581192,"[-0.5358244, 0.086343534, 0.16521375, 0.107873...",0.849551,0.833399,"[-0.06980602, 0.34916332, 0.1491546, -0.423699...",0.721071,0.659220
7,8,HR Senior Specialist,San Francisco Bay Area,500,1,hr senior specialist san francisco bay area,0.000000,0.000000,0.000000,0.000000,...,0.593965,"[0.0698105, 0.45483494, 0.05868009, 0.23186374...",0.465558,0.482086,"[-0.4082685, 0.48538813, 0.3310943, 0.05006316...",0.816862,0.807738,"[0.12938498, 0.20963639, 0.2089996, -0.3970928...",0.602913,0.656144
8,10,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,1,seek human resources hris generalist positions...,0.384900,0.471405,0.146901,0.235587,...,0.693475,"[0.070694834, -0.08755397, 0.059259363, 0.2419...",0.651652,0.712170,"[-0.45893243, 0.24516004, 0.3179749, 0.0450428...",0.849221,0.879234,"[-0.0074006296, 0.46556965, 0.51176244, -0.636...",0.683450,0.774203
9,11,Student at Chapman University,"Lake Forest, California",2,0,student chapman university lake forest california,0.000000,0.000000,0.000000,0.000000,...,0.532900,"[0.43293983, 0.06955156, 0.06899707, 0.1333234...",0.272591,0.217762,"[-0.5080098, 0.30833697, 0.49653065

In [20]:
df.to_csv("../data/vectors_hr_data.csv", index=False)